In [ ]:
# Replace 'YOUR_NOTEBOOK_NAME.ipynb' with the actual name of your Colab notebook
NOTEBOOK_NAME = "01_data_collection.ipynb" # Change this to your notebook's name

# Change to the repository directory
%cd {REPO_NAME}

# Copy the current notebook to the repository directory
!cp "/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}" .

# Add changes, commit, and push
!git add {NOTEBOOK_NAME}
!git commit -m "Add/Update {NOTEBOOK_NAME} from Colab"
!git push

%cd .. # Go back to the root content directory

print(f"Notebook '{NOTEBOOK_NAME}' pushed to '{REPO_NAME}'.")

/content/metro-transportation-analytics
cp: cannot stat '/content/01_data_collection.ipynb': No such file or directory
fatal: pathspec '01_data_collection.ipynb' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
[Errno 2] No such file or directory: '.. # Go back to the root content directory'
/content/metro-transportation-analytics
Notebook '01_data_collection.ipynb' pushed to 'metro-transportation-analytics'.


In [ ]:
# Replace 'YOUR_NOTEBOOK_NAME.ipynb' with the actual name of your Colab notebook
NOTEBOOK_NAME = "01_data_collection.ipynb" # Change this to your notebook's name

# Change to the repository directory
%cd {REPO_NAME}

# Copy the current notebook to the repository directory
!cp "/content/drive/MyDrive/Colab Notebooks/{NOTEBOOK_NAME}" .

# Add changes, commit, and push
!git add {NOTEBOOK_NAME}
!git commit -m "Add/Update {NOTEBOOK_NAME} from Colab"
!git push

%cd .. # Go back to the root content directory

print(f"Notebook '{NOTEBOOK_NAME}' pushed to '{REPO_NAME}'.")

/content/metro-transportation-analytics
cp: cannot stat '/content/01_data_collection.ipynb': No such file or directory
fatal: pathspec '01_data_collection.ipynb' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
[Errno 2] No such file or directory: '.. # Go back to the root content directory'
/content/metro-transportation-analytics
Notebook '01_data_collection.ipynb' pushed to 'metro-transportation-analytics'.


import censusdata

In [8]:
! pip install censusdata
import censusdata


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.6/26.6 MB 44.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for censusdata: filename=CensusData-1.15.post1-py3-none-any.whl size=28205744 sha256=78724c629b51aadbd2ff4c4ecb725aa16abf3569e7df5e3d56560c8faf99b5d8
  Stored in directory: /root/.cache/pip/wheels/54/5e/eb/518ccd7738e6b9b35d9fb3d226d45979066ec367ed26ad1369
Successfully built censusdata


In [3]:
from google.colab import drive
drive.mount('/content/drive',force_remount= True)

Mounted at /content/drive


In [5]:
!pip install pandas requests geopandas plotly


In [ ]:
import pandas as pd
from datetime import datetime
import time
from google.colab import userdata
import requests # Ensure requests is imported
from google.colab import drive

In [24]:

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

START_YEAR = 2016
END_YEAR = 2024

STATE_FIPS = "47"  # Tennessee

# Optional Census API Key
# Replace with your key if you have one
API_KEY = userdata.get("census_api")

# Check if API_KEY is empty
if not API_KEY:
    raise ValueError("Census API Key is missing. Please add it to Colab Secrets as 'census_api'.")

# ------------------------------------------------------------
# ACS VARIABLES
# ------------------------------------------------------------

VARIABLES = {
    "B01003_001E": "population",
    "B19013_001E": "median_household_income",
    "B25077_001E": "median_home_value",
    "B17001_002E": "poverty_population"
}

# ------------------------------------------------------------
# STORAGE
# ------------------------------------------------------------

all_data = []

# ------------------------------------------------------------
# YEAR LOOP
# ------------------------------------------------------------

for year in range(START_YEAR, END_YEAR + 1):

    print(f"\nProcessing Year: {year}")

    try:

        # ----------------------------------------------------
        # API URL
        # ----------------------------------------------------

        variables_string = ",".join(VARIABLES.keys())

        url = (
            f"https://api.census.gov/data/{year}/acs/acs5"
            f"?get=NAME,{variables_string}"
            f"&for=zip%20code%20tabulation%20area:*"
            f"&in=state:{STATE_FIPS}"
            f"&key={API_KEY}"
        )

        # ----------------------------------------------------
        # REQUEST
        # ----------------------------------------------------

        response = requests.get(url)

        # Print status code and response text for debugging
        print(f"API Response Status Code for {year}: {response.status_code}")
        print(f"API Response Text for {year}: {response.text}")

        if response.status_code != 200:
            print(f"Failed for {year}: {response.status_code}")
            continue

        # Check if the response is actually HTML (indicating an invalid key or error)
        if "<html" in response.text.lower() and "invalid key" in response.text.lower():
            print(f"✗ Error for {year}: API Key appears to be invalid or missing, received HTML response instead of JSON.")
            continue

        data = response.json()

        # ----------------------------------------------------
        # CONVERT TO DATAFRAME
        # ----------------------------------------------------

        df = pd.DataFrame(data[1:], columns=data[0])

        # ----------------------------------------------------
        # RENAME COLUMNS
        # ----------------------------------------------------

        df.rename(columns=VARIABLES, inplace=True)

        # ----------------------------------------------------
        # ADD YEAR
        # ----------------------------------------------------

        df["year"] = year

        # ----------------------------------------------------
        # EXTRACT COUNTY FROM NAME
        # ----------------------------------------------------
        # Example NAME:
        # "ZCTA5 37011"

        df["zip_code"] = df["zip code tabulation area"]

        # ----------------------------------------------------
        # COUNTY LOOKUP
        # ----------------------------------------------------
        # County is NOT directly available in ZIP API endpoint
        # We use ZIP -> County crosswalk later
        # Placeholder for now

        df["county"] = None

        # ----------------------------------------------------
        # STATE COLUMN
        # ----------------------------------------------------

        df["state"] = "Tennessee"

        # ----------------------------------------------------
        # KEEP COLUMNS
        # ----------------------------------------------------

        df = df[
            [
                "year",
                "state",
                "zip_code",
                "population",
                "median_household_income",
                "median_home_value",
                "poverty_population"
            ]
        ]

        # ----------------------------------------------------
        # CLEAN DATA TYPES
        # ----------------------------------------------------

        numeric_cols = [
            "population",
            "median_household_income"
        ]

        for col in numeric_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        all_data.append(df)

        print(f"✓ Success for {year}")

        # Avoid API rate limits
        time.sleep(1)

    except Exception as e:
        print(f"✗ Error for {year}: {e}")

# ------------------------------------------------------------
# COMBINE ALL YEARS
# ------------------------------------------------------------

if all_data:

    final_df = pd.concat(all_data, ignore_index=True)

    # --------------------------------------------------------
    # SORT
    # --------------------------------------------------------

    final_df.sort_values(
        by=["year", "zip_code"],
        inplace=True
    )

    # --------------------------------------------------------
    # SAVE CSV
    # --------------------------------------------------------

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    # Define the output path in Google Drive
    output_dir = "/content/drive/MyDrive/Census_Data"
    import os
    os.makedirs(output_dir, exist_ok=True)
    output_file = f"{output_dir}/tennessee_census_data_{timestamp}.csv"

    final_df.to_csv(output_file, index=False)

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    print("\n================================================")
    print("DATA PULL COMPLETE")
    print("================================================")
    print(f"Rows: {len(final_df)}")
    print(f"Output File: {output_file}")
    print("================================================")

    print("\nSample Data:")
    print(final_df.head())

else:
    print("No data downloaded.")


Processing Year: 2016
API Response Status Code for 2016: 200
API Response Text for 2016: [["NAME","B01003_001E","B19013_001E","B25077_001E","B17001_002E","state","zip code tabulation area"],
["ZCTA5 37323","30990","44881","122300","5012","47","37323"],
["ZCTA5 37325","1614","40323","116800","453","47","37325"],
["ZCTA5 37331","7832","34105","103900","1724","47","37331"],
["ZCTA5 37336","4493","51066","167800","474","47","37336"],
["ZCTA5 37359","600","35682","105300","114","47","37359"],
["ZCTA5 37367","11170","36707","110200","2406","47","37367"],
["ZCTA5 37405","15772","51826","224000","2356","47","37405"],
["ZCTA5 37407","10256","21627","63400","5128","47","37407"],
["ZCTA5 37409","2729","43448","144900","399","47","37409"],
["ZCTA5 37614","1945","-666666666","-666666666","0","47","37614"],
["ZCTA5 37645","5098","46523","130400","560","47","37645"],
["ZCTA5 37657","938","27639","76100","255","47","37657"],
["ZCTA5 37683","12908","28965","126800","3113","47","37683"],
["ZCTA5 37691"

### Step 1: Generate a GitHub Personal Access Token

To push to a GitHub repository from Colab, you need a Personal Access Token (PAT).

1.  Go to your GitHub settings: **Settings** > **Developer settings** > **Personal access tokens** > **Tokens (classic)**.
2.  Click **Generate new token (classic)**.
3.  Give it a descriptive name (e.g., `colab_repo_access`).
4.  Select the `repo` scope (this grants access to your repositories).
5.  Click **Generate token** and copy the generated token immediately. You won't see it again.

### Step 2: Store Your GitHub Token in Colab Secrets

For security, store your GitHub PAT in Colab's "Secrets" tab.

1.  In the left sidebar of Colab, click the **🔑 Secrets** icon.
2.  Click **+ New secret**.
3.  For **Name**, enter `GITHUB_TOKEN`.
4.  For **Value**, paste your GitHub Personal Access Token.
5.  Make sure to enable "Notebook access" for this secret.

### Step 3: Configure Git and Clone Your Repository

In [23]:
# Replace with your GitHub username and email
USER_NAME = "sanamica"
USER_EMAIL = "sanamica@gmail.com"

# Replace with the URL of your repository
REPO_URL = "https://github.com/sanamica/metro-transportation-analytics"
REPO_NAME = REPO_URL.split('/')[-1].replace('.git', '')

# Get the GitHub token from Colab secrets
GITHUB_TOKEN = userdata.get('github_token')

# Configure Git
!git config --global user.name "{USER_NAME}"
!git config --global user.email "{USER_EMAIL}"

# Clone the repository (if not already cloned)
import os
if not os.path.exists(REPO_NAME):
    !git clone https://{GITHUB_TOKEN}@github.com/{USER_NAME}/{REPO_NAME}.git
else:
    print(f"Repository '{REPO_NAME}' already exists. Pulling latest changes...")
    %cd {REPO_NAME}
    !git pull
    %cd ..

print(f"Successfully cloned or updated '{REPO_NAME}'.")

Cloning into 'metro-transportation-analytics'...
remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 10 (delta 1), reused 6 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (10/10), done.
Resolving deltas: 100% (1/1), done.
Successfully cloned or updated 'metro-transportation-analytics'.


### Step 4: Copy Your Notebook to the Repository and Push

Now, you need to copy your current notebook file into the cloned repository and then push the changes. You'll need to know the name of your current notebook file (e.g., `My_Census_Analysis.ipynb`). You can find this at the top of your Colab page.

In [25]:
# Replace 'YOUR_NOTEBOOK_NAME.ipynb' with the actual name of your Colab notebook
NOTEBOOK_NAME = "01_data_collection.ipynb" # Change this to your notebook's name

# Change to the repository directory
%cd {REPO_NAME}

# Copy the current notebook to the repository directory
!cp /content/{NOTEBOOK_NAME} .

# Add changes, commit, and push
!git add {NOTEBOOK_NAME}
!git commit -m "Add/Update {NOTEBOOK_NAME} from Colab"
!git push

%cd .. # Go back to the root content directory

print(f"Notebook '{NOTEBOOK_NAME}' pushed to '{REPO_NAME}'.")

/content/metro-transportation-analytics
cp: cannot stat '/content/01_data_collection.ipynb': No such file or directory
fatal: pathspec '01_data_collection.ipynb' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
[Errno 2] No such file or directory: '.. # Go back to the root content directory'
/content/metro-transportation-analytics
Notebook '01_data_collection.ipynb' pushed to 'metro-transportation-analytics'.
